# Art Therapy Recommender — GPU build on Colab

Builds the **world-art CLIP + FAISS index** on a free GPU (minutes, not the ~30 min CPU run at home). This dodges the macOS MPS / OpenMP segfaults entirely — Colab is Linux + CUDA.

**Steps:** GPU check → install deps → upload the project zip → ingest the balanced world corpus → embed on GPU → save `index_world` to Google Drive → download it.

> Before running: **Runtime → Change runtime type → T4 GPU**.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi -L
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

## 2. Install dependencies
Colab already ships torch+CUDA, so we only add the rest. `faiss-cpu` is fine — the GPU does the embedding; building a flat index is trivial on CPU.

In [ ]:
!pip install -q open_clip_torch faiss-cpu "datasets>=2.18" "pydantic>=2.6" pandas pyarrow tqdm requests
print('deps installed')

## 3. Upload the project code
Run the cell, then pick **`art-therapy-recommender-code.zip`** (the file I gave you). It unzips to `art-therapy-recommender/` and we `cd` in.

In [ ]:
import os, zipfile
from google.colab import files
uploaded = files.upload()  # choose art-therapy-recommender-code.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('.')
os.chdir('art-therapy-recommender')
print('now in:', os.getcwd())
print(os.listdir('.'))

## 4. (Optional) Mount Google Drive
So the finished index persists after the session ends. Skip if you only want to download it at the end.

In [ ]:
USE_DRIVE = True  #@param {type:'boolean'}
DRIVE_DIR = '/content/drive/MyDrive/art-therapy-recommender'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('will save index to', DRIVE_DIR)

## 5. Ingest the balanced world corpus
Pulls Met (global departments) + Cleveland + AIC + the 34 Indian art styles into one normalized corpus, and prints a coverage-by-region report. Tune `LIMIT` per source (300 ≈ ~1,200 works). The Met crawl is the slow part.

In [ ]:
LIMIT = 300  #@param {type:'integer'}
SOURCES = 'met,cleveland,aic,indian_art'  #@param {type:'string'}
!python -m ml.data.ingest --sources $SOURCES --limit $LIMIT --out data/processed/corpus_world.jsonl

## 6. Embed + build the FAISS index (GPU)
`ClipEncoder` auto-selects CUDA here, so this is the fast part. Writes `data/processed/index_world/` (`corpus.faiss`, `corpus_meta.jsonl`, `corpus_emb.npy`).

In [ ]:
!python -m ml.embeddings.build_index --corpus data/processed/corpus_world.jsonl --out data/processed/index_world --batch-size 64

## 7. Sanity check — a few mood + text queries

In [ ]:
from ml.embeddings.retrieval import MoodArtRetriever
r = MoodArtRetriever('data/processed/index_world')
print('index size:', len(r.meta))
for mood in ['joy', 'sadness', 'surprise']:
    print(f'\n--- mood: {mood} (therapeutic) ---')
    for a in r.by_mood(mood, k=4):
        print(f"  {a['_score']:.3f} [{a['region']:14}] [{a['source']:9}] {a['title'][:44]!r}")
print('\n--- text: peaceful indian temple sculpture ---')
for a in r.by_text('peaceful indian temple sculpture', k=4):
    print(f"  {a['_score']:.3f} [{a['region']:14}] [{a['source']:9}] {a['title'][:44]!r}")

## 8. Save the index (Drive + download)
Zips `index_world` (index + metadata; skips the big raw `.npy`), copies to Drive if mounted, and downloads it. Point your local `MoodArtRetriever` at the unzipped folder.

In [ ]:
import shutil, zipfile, glob
out_zip = 'index_world.zip'
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['corpus.faiss', 'corpus_meta.jsonl']:
        z.write(f'data/processed/index_world/{f}', f'index_world/{f}')
print('zipped ->', out_zip)
if 'USE_DRIVE' in dir() and USE_DRIVE:
    shutil.copy(out_zip, f'{DRIVE_DIR}/{out_zip}')
    print('copied to Drive:', f'{DRIVE_DIR}/{out_zip}')
from google.colab import files
files.download(out_zip)

---
### Back on your Mac
```bash
unzip index_world.zip -d data/processed/
python -m ml.embeddings.retrieval --index data/processed/index_world --mood joy
```
The retriever only needs `corpus.faiss` + `corpus_meta.jsonl` — no GPU to *query*, just to build.